
# DEAI – Classificatie Opdracht (Volledig Gecommentarieerd)

Dit notebook bevat:
- Binair model (Garage ja/nee)
- Multi-class model (Overall Qual)
- One-hot encoding
- Train/test split (horizontaal + verticaal)
- Hyperparameter experimenten
- Evaluatiemetrics
- Confusion matrices

Elke code-regel bevat uitleg in comments zodat je dit kunt toelichten tijdens je gesprek.


In [ ]:

import pandas as pd  # Voor het werken met tabellen (DataFrames)
import numpy as np  # Voor numerieke berekeningen
from sklearn.model_selection import train_test_split  # Om data te splitsen in train en test sets
from sklearn.preprocessing import OneHotEncoder  # Voor het omzetten van categorische variabelen naar getallen
from sklearn.compose import ColumnTransformer  # Om verschillende preprocessing stappen te combineren
from sklearn.pipeline import Pipeline  # Om preprocessing + model samen te voegen
from sklearn.tree import DecisionTreeClassifier  # Het gekozen classificatiemodel (Decision Tree)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay  # Evaluatiemetrics
import matplotlib.pyplot as plt  # Voor het visualiseren van de confusion matrix


In [ ]:

df = pd.read_excel('AmesHousing.xlsx', sheet_name='AmesHousing')  # Lees het juiste Excel tabblad in als DataFrame
print(df.columns.tolist())  # Toon alle kolomnamen om zeker te weten hoe ze exact heten
df.head()  # Toon eerste 5 rijen ter controle


In [ ]:

features = ['Overall Qual', 'Gr Liv Area', 'Neighborhood']  # Kies features (minstens 1 categorisch)
X = df[features]  # Maak feature matrix met alleen deze kolommen

y_bin = df['Garage'].map({'yes': 1, 'no': 0})  # Zet Garage om naar binaire target (1 = ja, 0 = nee)
y_multi = df['Overall Qual']  # Multi-class target: kwaliteitsniveau (1 t/m 10)


In [ ]:

categorical_features = ['Neighborhood']  # Categorische feature
numeric_features = ['Overall Qual', 'Gr Liv Area']  # Numerieke features

preprocessor = ColumnTransformer(  # Combineer verschillende bewerkingen
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),  # One-hot encode Neighborhood
        ('num', 'passthrough', numeric_features)  # Laat numerieke kolommen onveranderd
    ])


In [ ]:

X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(  # Split voor binaire classificatie
    X, y_bin, test_size=0.2, random_state=42)  # 80% training, 20% test

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(  # Split voor multi-class
    X, y_multi, test_size=0.2, random_state=42)  # Zelfde random_state voor reproduceerbaarheid


In [ ]:

model = DecisionTreeClassifier(max_depth=5, min_samples_split=10, random_state=42)  # Decision Tree met hyperparameters

pipeline = Pipeline(steps=[  # Maak pipeline zodat preprocessing + model samen trainen
    ('preprocessing', preprocessor),  # Eerst data transformeren
    ('classifier', model)  # Daarna model trainen
])

pipeline.fit(X_train_bin, y_train_bin)  # Train het model op trainingsdata

y_pred_bin = pipeline.predict(X_test_bin)  # Maak voorspellingen op testdata

acc_bin = accuracy_score(y_test_bin, y_pred_bin)  # Bereken accuracy
prec_bin = precision_score(y_test_bin, y_pred_bin)  # Bereken precision
rec_bin = recall_score(y_test_bin, y_pred_bin)  # Bereken recall
f1_bin = f1_score(y_test_bin, y_pred_bin)  # Bereken F1-score

print("Binair model resultaten:")
print(acc_bin, prec_bin, rec_bin, f1_bin)  # Toon evaluatie metrics


In [ ]:

cm_bin = confusion_matrix(y_test_bin, y_pred_bin)  # Maak confusion matrix
ConfusionMatrixDisplay(cm_bin).plot()  # Visualiseer confusion matrix
plt.title("Confusion Matrix - Binair Model")  # Titel toevoegen
plt.show()  # Toon plot


In [ ]:

pipeline.fit(X_train_multi, y_train_multi)  # Train pipeline opnieuw voor multi-class

y_pred_multi = pipeline.predict(X_test_multi)  # Maak voorspellingen

acc_multi = accuracy_score(y_test_multi, y_pred_multi)  # Accuracy
prec_multi = precision_score(y_test_multi, y_pred_multi, average='weighted')  # Weighted precision
rec_multi = recall_score(y_test_multi, y_pred_multi, average='weighted')  # Weighted recall
f1_multi = f1_score(y_test_multi, y_pred_multi, average='weighted')  # Weighted F1-score

print("Multi-class model resultaten:")
print(acc_multi, prec_multi, rec_multi, f1_multi)  # Toon metrics


In [ ]:

cm_multi = confusion_matrix(y_test_multi, y_pred_multi)  # Confusion matrix maken
ConfusionMatrixDisplay(cm_multi).plot()  # Visualiseren
plt.title("Confusion Matrix - Multi-class Model")  # Titel
plt.show()  # Plot tonen


In [ ]:

model_exp2 = DecisionTreeClassifier(max_depth=10, min_samples_split=5, random_state=42)  # Nieuwe hyperparameters

pipeline_exp2 = Pipeline(steps=[
    ('preprocessing', preprocessor),  # Zelfde preprocessing
    ('classifier', model_exp2)  # Nieuw model
])

pipeline_exp2.fit(X_train_bin, y_train_bin)  # Train opnieuw

y_pred_exp2 = pipeline_exp2.predict(X_test_bin)  # Nieuwe voorspellingen

print("Experiment 2 Accuracy:")
print(accuracy_score(y_test_bin, y_pred_exp2))  # Toon nieuwe accuracy
